In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

### Loading the Dataset:

In [2]:
df = pd.read_csv('D:/Important Files/Machine Learning/Datasets/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

### Confirming the Imbalance:

Worth quantifying exactly how imbalanced this dataset is before comparing the two models — this is the specific condition Complement NB is designed for.

In [3]:
counts = df['label'].value_counts()
imbalance_ratio = counts['ham'] / counts['spam']
print(f"Ham : Spam ratio = {imbalance_ratio:.2f} : 1")

Ham : Spam ratio = 6.46 : 1


### Preparing Word-Count Features (Same as Multinomial NB):

In [4]:
X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2, stratify=y)

vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

### Fitting Complement Naive Bayes:

In [5]:
cnb = ComplementNB()
cnb.fit(X_train_counts, y_train)

y_pred_cnb = cnb.predict(X_test_counts)

print("Accuracy:", accuracy_score(y_test, y_pred_cnb))
print("Precision:", precision_score(y_test, y_pred_cnb, pos_label='spam'))
print("Recall:", recall_score(y_test, y_pred_cnb, pos_label='spam'))
print("F1-Score:", f1_score(y_test, y_pred_cnb, pos_label='spam'))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_cnb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_cnb))

Accuracy: 0.9766816143497757
Precision: 0.9072847682119205
Recall: 0.9194630872483222
F1-Score: 0.9133333333333333

Confusion Matrix:
 [[952  14]
 [ 12 137]]

Classification Report:
               precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.91      0.92      0.91       149

    accuracy                           0.98      1115
   macro avg       0.95      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115



### Direct Comparison — Complement vs. Multinomial vs. Bernoulli:

Fitting Multinomial NB again on the identical split for a fair side-by-side, and pulling in Bernoulli's numbers from the previous notebook for a full three-way comparison.

In [6]:
mnb = MultinomialNB()
mnb.fit(X_train_counts, y_train)
y_pred_mnb = mnb.predict(X_test_counts)

# Bernoulli results from the previous notebook, same random_state/split, included here for a full comparison
bernoulli_results = {
    'Accuracy': 0.9766816143497757,
    'Precision (spam)': 0.992000,
    'Recall (spam)': 0.832215,
    'F1-Score (spam)': 0.905109
}

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (spam)', 'Recall (spam)', 'F1-Score (spam)'],
    'Complement NB': [
        accuracy_score(y_test, y_pred_cnb),
        precision_score(y_test, y_pred_cnb, pos_label='spam'),
        recall_score(y_test, y_pred_cnb, pos_label='spam'),
        f1_score(y_test, y_pred_cnb, pos_label='spam')
    ],
    'Multinomial NB': [
        accuracy_score(y_test, y_pred_mnb),
        precision_score(y_test, y_pred_mnb, pos_label='spam'),
        recall_score(y_test, y_pred_mnb, pos_label='spam'),
        f1_score(y_test, y_pred_mnb, pos_label='spam')
    ],
    'Bernoulli NB': [
        bernoulli_results['Accuracy'],
        bernoulli_results['Precision (spam)'],
        bernoulli_results['Recall (spam)'],
        bernoulli_results['F1-Score (spam)']
    ]
})

comparison

,Metric,Complement NB,Multinomial NB,Bernoulli NB
0,Accuracy,0.976682,0.982960,0.976682
1,Precision (spam),0.907285,0.964286,0.992000
2,Recall (spam),0.919463,0.906040,0.832215
3,F1-Score (spam),0.913333,0.934256,0.905109


### Stress-Testing on More Severe Imbalance:

The real advantage of Complement NB tends to show up more clearly as imbalance gets *more* severe than this dataset's ~6.5:1 ratio. To illustrate this, the cell below artificially creates a more imbalanced training set by subsampling the spam class down further, then compares both models' recall on the minority class under that harsher condition.

In [7]:
# Artificially shrink the spam class in the training set to simulate more severe imbalance
spam_train_idx = y_train[y_train == 'spam'].index
ham_train_idx = y_train[y_train == 'ham'].index

# Keep only 20% of spam training examples
np.random.seed(2)
reduced_spam_idx = np.random.choice(spam_train_idx, size=int(len(spam_train_idx) * 0.2), replace=False)
severe_train_idx = np.concatenate([ham_train_idx, reduced_spam_idx])

X_train_severe = X_train.loc[severe_train_idx]
y_train_severe = y_train.loc[severe_train_idx]

print("New training class balance:")
print(y_train_severe.value_counts())
print(f"New ratio: {(y_train_severe == 'ham').sum() / (y_train_severe == 'spam').sum():.2f} : 1")

X_train_severe_counts = vectorizer.transform(X_train_severe)

mnb_severe = MultinomialNB()
mnb_severe.fit(X_train_severe_counts, y_train_severe)
y_pred_mnb_severe = mnb_severe.predict(X_test_counts)

cnb_severe = ComplementNB()
cnb_severe.fit(X_train_severe_counts, y_train_severe)
y_pred_cnb_severe = cnb_severe.predict(X_test_counts)

print("\n--- Under severe imbalance ---")
print(f"Multinomial NB — Recall (spam): {recall_score(y_test, y_pred_mnb_severe, pos_label='spam'):.4f}")
print(f"Complement NB  — Recall (spam): {recall_score(y_test, y_pred_cnb_severe, pos_label='spam'):.4f}")

New training class balance:
label
ham     3859
spam     119
Name: count, dtype: int64
New ratio: 32.43 : 1

--- Under severe imbalance ---
Multinomial NB — Recall (spam): 0.7651
Complement NB  — Recall (spam): 0.8389


### Summary:

- Complement NB estimates word probabilities from the **complement** of each class (everything that isn't that class), then classifies by picking whichever complement model fits *worst*.
- Because the complement of a minority class is estimated from the much larger majority of the data, its parameter estimates are inherently more stable — this is the core reason Complement NB tends to outperform Multinomial NB as class imbalance grows more severe.
- On this dataset's original ~6.5:1 imbalance the two perform similarly, but the artificially more severe imbalance test above shows the gap Complement NB is specifically built to close.